# Automação de Busca de Vagas - Gupy

Projeto de automação com Selenium para busca e coleta de vagas no Gupy.

## 1. Configuração inicial

Importação de bibliotecas e configuração do navegador.

In [1]:

from selenium import webdriver
from selenium.webdriver.remote.webdriver import WebDriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from webdriver_manager.chrome import ChromeDriverManager
from urllib.parse import quote
from pathlib import Path
import pandas as pd


## 2. Iniciar o navegador

Criação da instância do navegador e acesso à página inicial do Gupy.

In [2]:
# criar o navegador
# Abre o navegador e acessa a página inicial do Gupy
servico = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=servico)
driver.get("https://www.gupy.io/")
driver.maximize_window()


## 3. Busca, extração e paginação

Função que busca vagas para um cargo, navega automaticamente por todas as páginas de resultado e extrai os dados de cada vaga encontrada.

In [3]:

def buscar_vagas(driver: WebDriver, termo: str) -> list:
    """
    Realiza a raspagem de vagas no Portal Gupy para uma localidade específica.

    Navega pelas páginas de resultados coletando informações detalhadas de cada 
    vaga disponível e trata possíveis instabilidades de carregamento ou fim de paginação.

    Args:
        driver (WebDriver): Instância ativa do navegador controlada pelo Selenium.
        termo (str): O cargo ou palavra-chave que será pesquisado.

    Returns:
        list[dict]: Uma lista de dicionários, onde cada dicionário contém as 
        informações de uma vaga (Título, Empresa, Local, Modelo, Tipo, Data, Link).
        Retorna uma lista vazia se nenhuma vaga for encontrada ou houver timeout.
    """
    # Monta a URL de busca já com o cargo e o filtro de localização codificados
    nome_vaga = quote(termo)

    nome_uf = "São Paulo"
    nome_uf = quote(nome_uf)

    nome_cidade = "São Paulo"
    nome_cidade = quote(nome_cidade)

    link_vaga = f"https://portal.gupy.io/job-search/term={nome_vaga}&state={nome_uf}&city[]={nome_cidade}"

    driver.get(link_vaga)

    lista_vagas = []

    # Percorre todas as páginas de resultado até não existir mais próxima página
    while True:
        try:
        
            try:
                # Espera os cards de vaga aparecerem; se nenhum aparecer a tempo, considera
                # que o cargo não teve resultado e devolve o que já foi coletado até aqui
                WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, 'a[href*="/job/"]')))
            except TimeoutException:
                print(f"Para o cargo '{termo}' não foram encontradas vagas ou a página falhou.")
                print(f"Aviso: Tempo limite atingido para o cargo {termo}.")
                return lista_vagas

            vagas = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/job/"]')

            for vaga in vagas:
                
                titulo = vaga.find_element(By.TAG_NAME, "h3").text

                empresas = vaga.find_elements(By.TAG_NAME, "p")

                # O card tem dois <p>: um é a empresa, o outro é a data de publicação
                # (identificados pelo prefixo fixo "Publicada em:")
                for empresa in empresas:
                    if not empresa.text.startswith("Publicada em:"):
                        nome_empresa = empresa.text
                    else:
                        data_vaga_publicada = empresa.text

                # Local é um campo opcional: pode não vir preenchido em algumas vagas
                local = vaga.find_elements(By.CSS_SELECTOR, 'span[data-testid="job-location"]')

                if len(local) == 0:
                    local_vaga = None
                else:
                    local_vaga = local[0].text

                # Modelo de trabalho, tipo de vaga e PcD não têm atributo próprio e são
                # opcionais, então são classificados pelo conteúdo do texto de cada span
                modelos_trabalho = [
                    "Presencial",
                    "Híbrido",
                    "Remoto"
                    ]

                tipos_vaga = [
                    "Estágio",
                    "Efetivo",
                    "Associado",
                    "Autônomo",
                    "Temporário",
                    "Pessoa Jurídica",
                    "Trainee",
                    "Sócio"
                    ]

                elementos_span = vaga.find_elements(By.TAG_NAME, "span")

                modelo_encontrado = None
                tipo_vaga_encontrada = None
                pcd_encontrado = None

                for el_span in elementos_span:
                    if el_span.text in modelos_trabalho:
                        modelo_encontrado = el_span.text

                    elif el_span.text in tipos_vaga:
                        tipo_vaga_encontrada = el_span.text

                    elif el_span.text == "Também p/ PcD":
                        pcd_encontrado = el_span.text
                        
                link = vaga.get_attribute("href")

                dic_vagas = {
                    "Cargo Buscado": termo,
                    "Titulo": titulo,
                    "Empresa": nome_empresa,
                    "Local": local_vaga,
                    "Modelo": modelo_encontrado,
                    "Tipo da Vaga": tipo_vaga_encontrada,
                    "Afirmativa para PcD": pcd_encontrado,
                    "Data": data_vaga_publicada,
                    "Link": link
                    }

                lista_vagas.append(dic_vagas)

            # Verifica se existe próxima página habilitada antes de tentar avançar
            proxima_pagina = driver.find_element(By.CSS_SELECTOR, 'button[aria-label="Próxima página"]')
            
            if proxima_pagina.is_enabled():
                proxima_pagina.click()
                try:
                    # Espera o conteúdo antigo sumir do DOM antes de considerar a página
                    # seguinte carregada, evitando StaleElementReferenceException
                    WebDriverWait(driver, 0).until(EC.staleness_of(vagas[0]))
                except TimeoutException:
                    print(f"Timeout ao carregar a próxima página para '{termo}'. Retornando o que foi coletado até aqui.")
                    return lista_vagas
            else:
                break

        except NoSuchElementException:
            print(f"Botão não encontrado. Fim das páginas para {termo}.")
            break           
            
    return lista_vagas


## 4. Execução para múltiplos cargos

Executa a busca para uma lista de cargos, juntando os resultados de todos em uma única lista.

In [4]:

cargos = [
    "Estágio TI",
    "Analista de Dados Júnior",
    "Desenvolvedor Júnior",
    "Analista de Automação Júnior",
    "Automaçao Júnior"
    ]

vagas_encontradas = []

for cargo in cargos:
# extend (e não append) porque cada chamada já devolve uma lista de vagas,
# mantendo tudo em uma única lista de dicionários, sem aninhamento
    vagas_encontradas.extend(buscar_vagas(driver, cargo))



Para o cargo 'Analista de Automação Júnior' não foram encontradas vagas ou a página falhou.
Aviso: Tempo limite atingido para o cargo Analista de Automação Júnior.
Para o cargo 'Automaçao Júnior' não foram encontradas vagas ou a página falhou.
Aviso: Tempo limite atingido para o cargo Automaçao Júnior.


## 5. Organização e exportação dos dados

Transformar os resultados em DataFrame e exportar para CSV.

In [ ]:
tabela_vagas = pd.DataFrame(vagas_encontradas)

# Remove o texto fixo da data e converte para datetime (dia primeiro,
# formato brasileiro), permitindo ordenar/filtrar por período depois
tabela_vagas["Data"] = tabela_vagas["Data"].str.replace("Publicada em:", "", regex=False).str.strip()

tabela_vagas["Data"] = pd.to_datetime(tabela_vagas["Data"], format='%d/%m/%Y')

# Preenche apenas os campos opcionais, mantendo a coluna Data intacta
colunas_nulos = [col for col in tabela_vagas.columns if col != "Data"]

for coluna in colunas_nulos:
    tabela_vagas[coluna] = tabela_vagas[coluna].fillna("Não informado") 

# Remove vagas repetidas entre buscas de cargos diferentes, usando o
# link como identificador único de cada vaga
tabela_vagas = tabela_vagas.drop_duplicates(subset=["Link"])

tabela_vagas.to_csv("reports/vagas_encontradas.csv", index=False, sep=";", encoding="utf-8-sig", date_format='%d/%m/%Y')


print(f"Total de vagas encontradas: {len(tabela_vagas)}")
#driver.quit()

NameError: name '__file__' is not defined